# Validating COMPASS on three OpenNeuro datasets

This notebook runs the COMPASS multi-agent inference engine end to end on three open clinical datasets and
checks how well it recovers each dataset's clinical phenotype. It loads the pre-built, blinded COMPASS
inputs, asserts every record is well formed, runs a 2-subject subset per dataset (each subject on one random
data tier) under a spend guard and shows the recovered phenotypes against ground truth, and estimates the
API cost and wall time of the full batch.

**Full-batch design (section 0.5).** The full validation routes each run across a verified five-provider
OpenRouter panel, balanced within every tier, and assigns each subject one data-complexity tier, balanced
across the cohort (a between-subjects ladder that reads the tier-vs-accuracy relationship at the cohort
level). The complete design contains 453 jobs: 100 intelligence, 143 psychosis, and 210 numeracy jobs
(105 subjects x 2 tasks). The panel table and exact per-dataset numbers are in sections 0.5 and 6.

Concurrency note: each engine run already fans out several tool/LLM calls internally
(`COMPASS_EXECUTOR_MAX_WORKERS`, default 12). Running many engine instances in an outer pool multiplies the
concurrent OpenRouter requests (outer x inner), which can exhaust a connection pool or provider rate limit.
The subset therefore runs one participant at a time with bounded internal fan-out. The full validation can
run one model-isolated process per panel provider, each with bounded internal fan-out, so routing remains
race-free while independent providers make progress in parallel.

| Dataset | Accession | Prediction output (phenotype structure) | Input: data-complexity tier ladder |
|---|---|---|---|
| INTELLIGENCE (AOMIC-ID1000) | ds003097 | total intelligence (univariate), then 3 IST subscales (multivariate) | 5 tiers: demographics, psychological, + morphometry, everything together, brain-only (morphometry + connectome) |
| PSYCHOSIS (first-episode) | ds003944 + ds003947 | diagnosis (binary), then BPRS total (univariate), then SAPS/SANS globals (multivariate) | 4 tiers: demographics + SES, clinical profile, full multimodal (+ 836 EEG), brain-only EEG signature |
| NUMERACY (stroke) | ds006533 | approximate and precise numeracy (two dissociable univariate phenotypes) | 4 tiers: demographics, aphasia, everything (per-parcel lesion), brain-only lesion |

Each dataset's exact tiers and phenotype structure are in its `PHENOTYPE_AND_TIERS.md` and are restated in
that dataset's own section below. **Models:** the subset runs use **deepseek/deepseek-v4-flash** (the
1M-context anchor); the full batch routes across the five-provider panel in section 0.5. **How to use:** the
simplest path is the one-click ALL THREE SUBSETS cell in section 5.1 (or Run All, top to bottom). You can
also just press a single dataset's SUBSET cell directly:
on a fresh kernel it auto-loads the setup cells above it first (free, no engine run), then runs its
2-subject test and shows the results inline. The FULL-BATCH cells at the end remain guarded against an
accidental Run All; the cost cell reports the estimate before an intentional launch.

**Automatic retry + resumability:** a failed participant is automatically retried once as a complete pipeline
run (the engine also retries individual API/tool calls internally). Every terminal result is cached to system
temp immediately, and a re-run skips successes while retrying only missing or failed runs. If a batch is
interrupted or trips the subset spend guard, run the same cell again and it continues without re-spending on
completed runs. Raw caches stay in system temp; the completed compact report can be exported into the repo.
Pass `rerun=True` to force a clean redo.

No target value (IST/BPRS/SAPS/SANS/numeracy) is ever an input: every phenotype is inferred only from the
multimodal predictor evidence, with native scales described to the engine in each record.

In [ ]:
import os, sys, json, io, time, shutil, contextlib, warnings, tempfile, urllib.request, importlib.util
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
warnings.filterwarnings("ignore")
# The engine parallelizes each run internally (its plan executor fans out up to COMPASS_EXECUTOR_MAX_WORKERS
# tool calls at once, embeddings up to COMPASS_EMBEDDING_MAX_WORKERS). On large tiers, 12 concurrent
# large-context calls exhaust the connection pool / hit the OpenRouter rate limit and stall, so we bound
# the internal fan-out and run one participant at a time (outer pool = 1): total concurrent provider calls
# stays near 3, which runs smoothly. For the full batch, raise these only as far as your rate limit allows.
os.environ.setdefault("COMPASS_EXECUTOR_MAX_WORKERS", "3")
os.environ.setdefault("COMPASS_EMBEDDING_MAX_WORKERS", "8")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
try:  # render figures inline when in a notebook; harmless no-op as a plain script
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

# Repo root = the folder containing src/full_stack (works from any working directory / PyCharm).
REPO = Path.cwd().resolve()
while not (REPO / "src" / "full_stack").is_dir():
    if REPO == REPO.parent:
        raise RuntimeError("Could not locate repo root (src/full_stack) above the working directory")
    REPO = REPO.parent
for p in (str(REPO), str(REPO / "validation")):
    if p not in sys.path:
        sys.path.insert(0, p)
VAL = REPO / "validation" / "datasets"

# All engine scratch (input copies, outputs, logs, cached results) lives in the SYSTEM temp dir, never
# inside the repo, so nothing here is ever committed.
SCRATCH = Path(tempfile.gettempdir()) / "compass_openneuro_validation"
(SCRATCH / "outputs").mkdir(parents=True, exist_ok=True)
(SCRATCH / "logs").mkdir(parents=True, exist_ok=True)

# DEFAULT_MODEL is the 1M-context anchor used by the cheap SUBSET sanity runs and the load-check. The
# FULL-batch design (section 0.5 and sections 6-7) does NOT use one model: it routes each run across a
# verified five-provider panel (MODEL_PANEL, defined in section 0.5) to cut cost and spread the rate limit.
DEFAULT_MODEL = "deepseek/deepseek-v4-flash"   # cheap, 1M context; the anchor for the subset run + load-check
DESIGN_SEED = 20260723                         # one seed drives BOTH randomization layers (fully reproducible)
N_PER_DATASET = 2                              # subjects per dataset for the subset sanity run
MAX_WORKERS = 1                                # OUTER runs stay sequential: model routing uses process-global settings.
SUBSET_MAX_USD = 8.0                           # subset-only guard; FULL batches intentionally have no dollar cap
MAX_ITERATIONS = 1                             # actor-critic depth per run
JOB_MAX_ATTEMPTS = 2                           # whole-pipeline attempts; catches failures before step auto-repair
RETRY_BACKOFF_S = 2                            # pause before a whole-pipeline retry
RUN_HEARTBEAT_S = 30                           # visible liveness update during multi-minute runs
VALIDATION_HELPERS_VERSION = "2026-07-24-live-verbose-v8"  # lets execution cells refresh stale kernels

from src.full_stack.backend.config.settings import LLMBackend, get_settings
S = get_settings()
S.models.backend = LLMBackend.OPENROUTER
DETERMINISTIC_TEMPERATURE = 0.0
for _role in ("orchestrator", "critic", "predictor", "integrator", "communicator", "tool"):
    setattr(S.models, f"{_role}_temperature", DETERMINISTIC_TEMPERATURE)

def set_model(model_id):
    """Point every agent role at one OpenRouter model.

    get_settings() returns a process singleton, so this changes the model the NEXT run_compass_pipeline call
    will use. We run the outer pool sequentially (MAX_WORKERS=1), so setting the model per run just before it
    starts is race-free and is how the full-batch design routes each run to its assigned provider.
    """
    S.models.public_model_name = model_id
    for role in ("orchestrator", "critic", "predictor", "integrator", "communicator", "tool"):
        setattr(S.models, f"{role}_model", model_id)

set_model(DEFAULT_MODEL)
S.paths.output_dir = SCRATCH / "outputs"
S.paths.logs_dir = SCRATCH / "logs"

import ssl
try:
    import certifi
    _SSL_CTX = ssl.create_default_context(cafile=certifi.where())   # Framework Python lacks system CAs
except Exception:
    _SSL_CTX = ssl.create_default_context()

def openrouter_usage():
    "Total OpenRouter USD used so far (None if unavailable)."
    key = S.openrouter_api_key
    if not key:
        return None
    try:
        req = urllib.request.Request("https://openrouter.ai/api/v1/credits",
                                     headers={"Authorization": f"Bearer {key}"})
        with urllib.request.urlopen(req, timeout=15, context=_SSL_CTX) as r:
            return float(json.loads(r.read().decode()).get("data", {}).get("total_usage", 0.0))
    except Exception:
        return None

assert S.openrouter_api_key, "OPENROUTER_API_KEY missing from repo-root .env"
_u = openrouter_usage()
print(f"repo:    {REPO}")
print(f"scratch: {SCRATCH}  (system temp, never committed)")
print(f"subset anchor model: {DEFAULT_MODEL}  |  workers: {MAX_WORKERS}  |  temperature: {DETERMINISTIC_TEMPERATURE:.1f}  |  subset spend cap: +${SUBSET_MAX_USD:.2f}")
print(f"full-batch routing:  5-provider panel (see section 0.5)  |  design seed: {DESIGN_SEED}  |  spend cap: NONE")
print(f"OpenRouter usage so far: {'$'+format(_u,'.4f') if _u is not None else 'unavailable'}")

## 0.5 Design: a five-provider panel and two balancing layers

The full batch makes two design choices, plus one cohort choice. This section defines them; section 6 gives
the exact per-dataset cost.

### The verified five-provider panel

The full batch spreads runs across five OpenRouter providers rather than one model. Each is fast, recent,
and priced at or below the panel's most expensive model (`deepseek/deepseek-v4-flash`, the 1M-context anchor
also used for the subset). Every field below was checked live against OpenRouter's endpoints API on
2026-07-23 (id, required host, precision, exact context, price per 1M tokens, release):

| Model (OpenRouter id) | Host | Precision | Context | In $/1M | Out $/1M | Released |
|---|---|---|---|---|---|---|
| deepseek/deepseek-v4-flash | DeepInfra | FP4 | 1,048,576 | 0.090 | 0.180 | 2026-04-24 |
| poolside/laguna-xs-2.1 | Poolside | FP8 | 262,144 | 0.060 | 0.120 | 2026-07-02 |
| nex-agi/nex-n2-mini | Nex AGI | FP8 | 262,144 | 0.025 | 0.100 | 2026-06-24 |
| qwen/qwen3.5-9b | DeepInfra | BF16 | 262,144 | 0.100 | 0.150 | 2026-03-10 |
| openai/gpt-oss-120b | DeepInfra | BF16 | 131,072 | 0.037 | 0.170 | 2025-08-05 |

Notes. Prices are the verified cheapest/host endpoint. `qwen3.5-9b` is the one model whose input list price
(0.10) sits just above the anchor (0.09); it is kept for its BF16 quality, and its output (0.15) is below
the anchor's (0.18), so on our input-heavy blend it stays competitive. Context is a non-issue: our largest
single record is about 21k tokens (psychosis full multimodal, all 836 EEG features), far under the smallest
panel context (131,072), so every model can serve every tier with no truncation. Our workload is about 92%
input tokens (multi-agent system prompts + reasoning), so the price that matters is roughly
`0.92*input + 0.08*output`; on that blend the panel averages about **$0.069/1M**. Spreading across five
hosts also avoids making one host a single point of failure. Outer engine runs remain sequential because
the engine's per-role model routing is process-global; internal tool calls still run concurrently.

### The two balancing layers (both seeded by `DESIGN_SEED`)

1. **Provider balancing.** Each run is assigned one panel model, cycled so that within every tier all five
   providers appear about equally. Provider identity is therefore decorrelated from tier: a tier-vs-accuracy
   effect can never be an artifact of one provider, and no host is a single point of failure.

2. **Tier balancing across subjects.** Each subject is assigned one data-complexity tier, with the tiers
   balanced across the cohort (a between-subjects ladder). This reads the full tier-vs-accuracy dose-response
   at the cohort level from `n_cohort` runs (one per subject), across subjects rather than within each one.

The result (section 6) is a **453-job** full validation: 100 intelligence jobs, 143 psychosis jobs, and
210 numeracy jobs (105 subjects x 2 tasks). Each ladder carries its non-redundant tiers (intelligence 5,
psychosis 4, numeracy 4).

In [ ]:
# ---- The verified five-provider panel (every figure confirmed live on OpenRouter's endpoints API, 2026-07-23) ----
# Each model is at or below our deepseek anchor's price, fast, and recent. Prices are USD per 1M tokens.
MODEL_PANEL = [
    dict(id="deepseek/deepseek-v4-flash", host="DeepInfra", precision="FP4",  context=1_048_576, in_usd=0.090, out_usd=0.180, released="2026-04-24"),
    dict(id="poolside/laguna-xs-2.1",     host="Poolside",  precision="FP8",  context=262_144,   in_usd=0.060, out_usd=0.120, released="2026-07-02"),
    dict(id="nex-agi/nex-n2-mini",        host="Nex AGI",   precision="FP8",  context=262_144,   in_usd=0.025, out_usd=0.100, released="2026-06-24"),
    dict(id="qwen/qwen3.5-9b",            host="DeepInfra", precision="BF16", context=262_144,   in_usd=0.100, out_usd=0.150, released="2026-03-10"),
    dict(id="openai/gpt-oss-120b",        host="DeepInfra", precision="BF16", context=131_072,   in_usd=0.037, out_usd=0.170, released="2025-08-05"),
]
PANEL_BY_ID = {m["id"]: m for m in MODEL_PANEL}

# Our runs are heavily input-weighted (deepseek's own activity: ~535B prompt vs ~44B output-billed tokens,
# so ~92% input), which comes from the multi-agent system prompts + reasoning. The price that matters is
# therefore an input-weighted blend of the per-1M input/output prices, not the headline output price.
INPUT_SHARE = 0.92
def blended_price(m):
    return INPUT_SHARE * m["in_usd"] + (1.0 - INPUT_SHARE) * m["out_usd"]
_ANCHOR_BLEND = blended_price(PANEL_BY_ID[DEFAULT_MODEL])
def model_price_factor(model_id):
    "Blended price of `model_id` relative to the deepseek anchor (1.0 = same, <1 = cheaper)."
    return blended_price(PANEL_BY_ID[model_id]) / _ANCHOR_BLEND

import random
def assign_design(subject_ids, tiers, seed=DESIGN_SEED):
    """Two seeded balancing layers over a cohort. Returns [{subject, tier, model}, ...].

    Layer 2 (optimization 2): each subject gets ONE data-complexity tier, tiers balanced across subjects
    (a between-subjects ladder instead of predicting every tier per subject).
    Layer 1 (optimization 1): each subject gets ONE panel model, cycled with a per-tier offset so that
    within every tier all five providers appear about equally. This crosses tier x provider, so provider
    identity is never confounded with tier and no single provider is a bottleneck.
    Both layers are driven by the one seed, so the whole plan is reproducible.
    """
    rng = random.Random(seed)
    subs = list(subject_ids); rng.shuffle(subs)
    n_t, n_m = len(tiers), len(MODEL_PANEL)
    per_tier = {t: 0 for t in tiers}
    out = []
    for i, s in enumerate(subs):
        tier = tiers[i % n_t]                                       # opt 2: balanced tiers across subjects
        j = per_tier[tier]; per_tier[tier] += 1
        model = MODEL_PANEL[(tiers.index(tier) + j) % n_m]["id"]     # opt 1: providers staggered per tier
        out.append(dict(subject=s, tier=tier, model=model))
    return out

def design_balance_table(design, tiers):
    "tier x provider contingency table for a design (rows = tier, cols = short model id, last col = ALL)."
    short = {m["id"]: m["id"].split("/")[-1] for m in MODEL_PANEL}
    tab = pd.DataFrame(0, index=list(tiers), columns=[short[m["id"]] for m in MODEL_PANEL])
    for a in design:
        tab.loc[a["tier"], short[a["model"]]] += 1
    tab["ALL"] = tab.sum(axis=1)
    return tab

# Show the panel economics, then prove the crossing is balanced on a synthetic 300-subject x 6-tier cohort.
print("Five-provider panel (verified 2026-07-23) with input-weighted blended price and factor vs deepseek:")
for m in MODEL_PANEL:
    print(f"  {m['id']:32} {m['host']:9} {m['precision']:4} ctx={m['context']:>9,}  "
          f"in/out ${m['in_usd']:.3f}/{m['out_usd']:.3f}  blend ${blended_price(m):.4f}/1M  "
          f"factor {model_price_factor(m['id']):.2f}  ({m['released']})")
print(f"\nPanel mean price factor vs deepseek-only: {np.mean([model_price_factor(m['id']) for m in MODEL_PANEL]):.3f}  "
      f"(panel mean blend ${np.mean([blended_price(m) for m in MODEL_PANEL]):.4f}/1M vs deepseek ${_ANCHOR_BLEND:.4f}/1M)")
_demo_tiers = [f"T{k}" for k in range(6)]
_demo = assign_design([f"s{i:03d}" for i in range(300)], _demo_tiers)
print("\nBalancing self-test (300 synthetic subjects, 6 tiers): tier x provider counts stay near-uniform:")
print(design_balance_table(_demo, _demo_tiers).to_string())

## 1. Dataset registry

Each dataset contributes one or more prediction TASKS (psychosis and intelligence each have a single
hierarchical task; numeracy has two univariate tasks, one per numeracy system). For every dataset we hold
its full tier ladder, the full-complexity tier used for the subset run, the two chosen subjects, and each
task's `PredictionTaskSpec` + global instruction (native scales, imported from the dataset's own pipeline
module so the engine stays dataset-agnostic). Ground truth is loaded for scoring only, never shown to the
engine.

In [ ]:
def _load(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); sys.modules[name] = m
    spec.loader.exec_module(m); return m

from src.full_stack.backend.data.models.prediction_task import (
    PredictionMode, PredictionTaskNode, PredictionTaskSpec)

DATASETS = {}   # key -> dict(label, tiers, full_tier, tasks, subjects, n_cohort, src_dir, ground_truth, diagnosis)

# ---------------- INTELLIGENCE (AOMIC-ID1000) ----------------
ist = _load(VAL / "INTELLIGENCE" / "pipeline" / "config.py", "ist_config")
ist_ann = {a["participant_id"]: a for a in
           json.load(open(VAL / "INTELLIGENCE" / "results" / "annotations.json"))["annotations"]}
_part = pd.read_csv(VAL / "INTELLIGENCE" / "dataset" / "participants.tsv", sep="\t", na_values=["n/a", "N/A", ""])
_ist_outputs = [ist.TARGET["column"]] + [s["column"] for s in ist.SUBSCALES]
_ist_stats = {c: {"mean": round(float(pd.to_numeric(_part[c], errors="coerce").mean()), 2),
                  "sd": round(float(pd.to_numeric(_part[c], errors="coerce").std(ddof=0)), 2)} for c in _ist_outputs}

def _ist_spec():
    return PredictionTaskSpec(task_id="ist_hierarchical", root=PredictionTaskNode(
        node_id="total_intelligence", display_name=ist.TARGET["label"],
        mode=PredictionMode.UNIVARIATE_REGRESSION, regression_outputs=[ist.TARGET["column"]],
        unit_by_output={ist.TARGET["column"]: "IST points"},
        children=[PredictionTaskNode(node_id="ist_subscales",
            display_name="IST subscales: fluid, memory, crystallised",
            mode=PredictionMode.MULTIVARIATE_REGRESSION, regression_outputs=[s["column"] for s in ist.SUBSCALES],
            unit_by_output={s["column"]: "IST points" for s in ist.SUBSCALES})]))

def _ist_gi():
    L = [ist.DATASET_CONTEXT, "", ist.IST_CONTEXT, "",
         "Predict these related scores on their NATIVE IST scales:"]
    t = _ist_stats[ist.TARGET["column"]]
    L.append(f"- {ist.TARGET['column']} ({ist.TARGET['label']}): overall composite. "
             f"Population reference mean={t['mean']}, sd={t['sd']} native IST points.")
    for s in ist.SUBSCALES:
        st = _ist_stats[s["column"]]
        L.append(f"- {s['column']} ({s['label']}): {s['description']} mean={st['mean']}, sd={st['sd']}.")
    L += ["", "Subscales are components of the total and move with it. No IST value for this participant is "
          "provided; infer every score only from the non-cognitive multimodal evidence. Return one numeric "
          "value per output on its native scale."]
    return "\n".join(L)

_ist_sorted = sorted(ist_ann.values(), key=lambda a: a["ground_truth"][ist.TARGET["column"]])
DATASETS["INTELLIGENCE"] = dict(
    label="AOMIC-ID1000 (ds003097): intelligence",
    tiers=["T1_demographics", "T2_psychological", "T3_morphometry", "T4_multimodal_full", "T5_brain_only"],
    full_tier="T4_multimodal_full", subset_tier="T2_psychological", n_cohort=len(ist_ann),
    tasks=[dict(name="ist", spec=_ist_spec(), gi=_ist_gi(), outputs=_ist_outputs,
                target_label=ist.TARGET["label"], control_label="")],
    subjects=[_ist_sorted[0]["participant_id"], _ist_sorted[-1]["participant_id"]][:N_PER_DATASET],
    src_dir=lambda tier, subj, task: VAL / "INTELLIGENCE" / "compass_inputs" / tier / subj,
    ground_truth=lambda subj, task: ist_ann[subj]["ground_truth"],
    diagnosis=lambda subj: None)

# ---------------- PSYCHOSIS (first-episode) ----------------
sys.path.insert(0, str(VAL / "PSYCHOSIS_FIRST_EPISODE"))
from utils import compass_task as PSY
_psy_ec = json.load(open(VAL / "PSYCHOSIS_FIRST_EPISODE" / "results" / "compass" / "eval_cohort.json"))
_psy_annfile = json.load(open(VAL / "PSYCHOSIS_FIRST_EPISODE" / "results" / "compass" / "annotations.json"))
_psy_ann = {a["recording_id"]: a for a in _psy_annfile["annotations"]}
_psy_gi = _psy_ec["global_instruction"] + (
    "\n\nHard output constraints: return a finite numeric prediction for the BPRS total and "
    "for every one of the four SAPS and five SANS outputs—never omit an output. BPRS "
    "is a 19-item sum constrained to 19-133 (never an average); all SAPS/SANS globals "
    "must stay within 0-5 inclusive."
)
DATASETS["PSYCHOSIS"] = dict(
    label="First-episode psychosis (ds003944+ds003947)",
    tiers=["T1_demographic_socioeconomic", "T2_clinical_profile", "T3_multimodal_full", "T4_eeg_brain_only"],
    full_tier="T3_multimodal_full", subset_tier="T2_clinical_profile",
    n_cohort=_psy_annfile["n_recordings"],  # full dataset = 143 recordings
    tasks=[dict(name="psy", spec=PSY.build_task_spec(), gi=_psy_gi,
                outputs=PSY.ALL_OUTPUTS, target_label=PSY.CASE_LABEL, control_label=PSY.CONTROL_LABEL,
                valid_labels=[PSY.CONTROL_LABEL, PSY.CASE_LABEL],
                valid_ranges={PSY.BPRS_TOTAL: (19, 133),
                              **{o: (0, 5) for o in [*PSY.SAPS_GLOBALS.values(), *PSY.SANS_GLOBALS.values()]}})],
    subjects=[_psy_ec["psychosis"][0], _psy_ec["control"][0]][:N_PER_DATASET],  # 1 case + 1 control
    src_dir=lambda tier, subj, task: VAL / "PSYCHOSIS_FIRST_EPISODE" / "results" / "compass" / "inputs" / tier / subj,
    ground_truth=lambda subj, task: _psy_ann[subj]["ground_truth"],
    diagnosis=lambda subj: _psy_ann[subj]["diagnosis"])

# ---------------- NUMERACY (stroke) ----------------
num = _load(VAL / "NUMERACY_STROKE" / "pipeline" / "compass_task.py", "num_task")
_num_tasks, _num_gt = [], {}
for _t in num.TARGETS:
    _m, _s = num.reference_stats(_t)
    _num_tasks.append(dict(name=_t, spec=num.build_task_spec(_t), gi=num.build_global_instruction(_t, _m, _s),
                           outputs=[_t], target_label=num.TARGET_LABEL[_t], control_label="",
                           valid_ranges={_t: (-5, 5)}))
    sub = json.load(open(VAL / "NUMERACY_STROKE" / "results" / f"subset_{_t}.json"))
    _num_gt[_t] = {p["participant_id"]: {_t: p["ground_truth"]} for p in sub["participants"]}
_num_blind = VAL / "NUMERACY_STROKE" / "compass_inputs"
def _num_present(subj):
    return all((_num_blind / f"{num.TARGET_SHORT[t]}_{lv}_blinded" / subj).is_dir()
               for t in num.TARGETS for lv in num.LEVELS)
DATASETS["NUMERACY"] = dict(
    label="Numeracy after stroke (ds006533)",
    tiers=num.LEVELS, full_tier="T3_lesion_fine", subset_tier="T3_lesion_fine",
    n_cohort=json.load(open(VAL / "NUMERACY_STROKE" / "results" / "annotations.json"))["n_subjects"],  # 105
    tasks=_num_tasks,
    subjects=[s for s in sorted(_num_gt[num.TARGETS[0]]) if _num_present(s)][:N_PER_DATASET],
    src_dir=lambda tier, subj, task: _num_blind / f"{num.TARGET_SHORT[task]}_{tier}_blinded" / subj,
    ground_truth=lambda subj, task: _num_gt[task][subj],
    diagnosis=lambda subj: None)

for k, d in DATASETS.items():
    n = len(d["subjects"]) * len(d["tasks"])
    print(f"{k:12} subset tier={d['subset_tier']:26} ({n} runs) | full tier={d['full_tier']:20} "
          f"| ladder={len(d['tiers'])} tiers, cohort~{d['n_cohort']} | subjects={d['subjects']}")

In [ ]:
# ---- Full-validation cohorts (used by the section 6 estimate and the guarded section 7 runs) ----
# The SUBSET cells (sections 3-5) run d["subjects"] (2 built subjects) on one tier with DEFAULT_MODEL. The
# FULL batch instead runs a between-subjects design over d["full_cohort"]: one tier + one provider per
# subject, assigned by assign_design(). We attach that cohort and a full-cohort ground-truth resolver here.

# INTELLIGENCE: use the complete leakage-safe 100-person evaluation cohort. All five tiers already exist
# for these blinded eval IDs and annotations.json carries every hierarchical IST ground-truth output.
_ist_cohort = sorted(ist_ann)
DATASETS["INTELLIGENCE"].update(
    full_cohort=_ist_cohort, n_cohort=len(_ist_cohort),
    full_gt=lambda subj, task: ist_ann[subj]["ground_truth"], full_diag=lambda subj: None,
    full_src_dir=lambda tier, subj, task: VAL / "INTELLIGENCE" / "compass_inputs" / tier / subj)

# PSYCHOSIS: the full 143 recordings already carry ground truth + diagnosis in annotations.json.
DATASETS["PSYCHOSIS"].update(
    full_cohort=list(_psy_ann.keys()), n_cohort=len(_psy_ann),
    full_gt=lambda subj, task: _psy_ann[subj]["ground_truth"],
    full_diag=lambda subj: _psy_ann[subj]["diagnosis"],
    full_src_dir=lambda tier, subj, task: SCRATCH / "full_inputs" / "PSYCHOSIS" / tier / subj)

# NUMERACY: the full 105 subjects carry both-phenotype ground truth in annotations.json.
_num_annfile = json.load(open(VAL / "NUMERACY_STROKE" / "results" / "annotations.json"))
_num_gt_full = {a["participant_id"]: a["ground_truth"] for a in _num_annfile["annotations"]}
DATASETS["NUMERACY"].update(
    full_cohort=[a["participant_id"] for a in _num_annfile["annotations"]], n_cohort=len(_num_gt_full),
    full_gt=lambda subj, task: {task: _num_gt_full[subj][task]}, full_diag=lambda subj: None,
    full_src_dir=lambda tier, subj, task: _num_blind / f"{num.TARGET_SHORT[task]}_{tier}_allshared" / subj)

# Report each full cohort and how many already have blinded inputs built on disk (runnable now vs to-build).
def _built_at(dkey, tier):
    d = DATASETS[dkey]
    resolve = d.get("full_src_dir", d["src_dir"])
    return sum(1 for s in d["full_cohort"] if resolve(tier, s, d["tasks"][0]["name"]).is_dir())
print("Full-validation cohorts (between-subjects design => one tier per subject):")
for k, d in DATASETS.items():
    old_runs = d["n_cohort"] * len(d["tasks"]) * len(d["tiers"])
    new_runs = d["n_cohort"] * len(d["tasks"])
    print(f"  {k:12} n_cohort={d['n_cohort']:>3} | tiers={len(d['tiers'])} | tasks={len(d['tasks'])} | "
          f"runs {old_runs:>4} -> {new_runs:>3} | built@subset_tier={_built_at(k, d['subset_tier'])}")

## 1.1 The data-complexity ladders and the phenotype targets

Every run predicts a dataset's clinical phenotype from a blinded record; the only thing that changes up a
ladder is HOW MUCH evidence the engine sees. Each ladder builds up to everything together, then ends with a
brain-only tier that isolates the neural/anatomical contribution.

**Data-complexity tier ladders** (each tier adds a modality; the target is never an input)

| INTELLIGENCE (AOMIC-ID1000), 5 tiers | evidence |
|---|---|
| T1_demographics | age, sex, handedness, BMI, education, socio-economic status |
| T2_psychological | + NEO-FFI Big Five, BIS/BAS, STAI, sexual/gender identity, religiosity (all self-report) |
| T3_morphometry | + FreeSurfer brain morphometry |
| T4_multimodal_full (everything) | + movie-fMRI functional connectome (all modalities together) |
| T5_brain_only | brain only: morphometry + connectome (no self-report) |

The self-report questionnaires and identity/belief are merged into one psychological tier; the ladder then
climbs to everything together (T4) and ends with a brain-only tier (T5).

| PSYCHOSIS (ds003944 + ds003947), 4 tiers | evidence |
|---|---|
| T1_demographic_socioeconomic | demographics + Hollingshead socio-economic status |
| T2_clinical_profile | + MATRICS cognition, WASI IQ, GAF/SFS observed functioning |
| T3_multimodal_full (everything) | + all 836 resting-EEG features |
| T4_eeg_brain_only | brain only: 79-feature psychosis-signature resting EEG |

| NUMERACY (ds006533), 4 tiers | evidence |
|---|---|
| T1_demographics | age, education, imaging modality |
| T2_aphasia | + WAB-R aphasia quotient, whole-brain lesion load |
| T3_lesion_fine (everything) | + per-parcel lesion overlap (194 features) |
| T4_lesion_brain_only | brain only: per-parcel lesion overlap (189 features, no demographics/clinical) |

**Complete phenotype target structure** (the hierarchy the engine predicts per run)

```text
INTELLIGENCE  (hierarchical regression)
  total_intelligence          univariate     IST_intelligence_total          native IST 2000-R points
    ist_subscales             multivariate   IST_fluid, IST_memory, IST_crystallised

PSYCHOSIS     (mixed-type hierarchy)
  diagnosis                   binary         Control vs First-Episode Psychosis
    global_severity           univariate     BPRS 19-item total              19-133
      positive_symptoms       multivariate   4 SAPS global ratings           0-5 each
      negative_symptoms       multivariate   5 SANS global ratings           0-5 each

NUMERACY      (two dissociable univariate phenotypes, predicted separately)
  approximate_numeracy        univariate     non-symbolic Approximate Number System   population Z-score
  precise_numeracy            univariate     precise symbolic numeracy                population Z-score
```

The engine is told each native scale (and, for numeracy, that the transformed tier is standardized) in the
per-dataset global instruction and in each record's text. A result is rejected before caching if a required
value is missing/non-finite or outside the declared task scale (NUMERACY population Z: -5 to +5; BPRS:
19-133; SAPS/SANS globals: 0-5), so structurally valid but uninterpretable outputs are retried. Full per-dataset
detail is in each dataset's `PHENOTYPE_AND_TIERS.md`.

## 2. Engine run helpers and load-check

`run_job` copies a record's four files into a uniquely named participant directory (in system temp) so
workers never collide on the engine's per-participant output path, runs the full COMPASS pipeline, and
harvests the prediction from the in-memory result. Entry points share one resumable runner
(`_run_batch`): `run_smoketest` runs the 2-subject subset with each subject on one random tier (deepseek
anchor), and `run_full_design` runs the FULL batch under the section 0.5 design (`build_design_jobs` assigns
one balanced tier and one balanced provider per subject). Each run is routed to its assigned model by `set_model`
just before it starts (safe because the outer runner is sequential), and every result row records which
provider served it. Every subject streams the detailed COMPASS stage/tool/agent output live while also
retaining a failure tail. Failed participant pipelines automatically retry once, long attempts emit
30-second heartbeats, and full batches first probe every panel model with a tiny live completion. Subsets
retain the +$8 guard; full-batch runs have no dollar cap and only report usage. Required labels, finite
regression outputs, and declared native-scale bounds are validated before a result enters the cache. The load-check then
asserts every subset record parses through the engine's own
`DataLoader` with a valid ontology and no target leakage.

In [ ]:
def harvest(result):
    "Generic: pull the diagnosis label/probs and every regression value from any hierarchy."
    pred = (result.get("internal_context") or {}).get("prediction")
    root = getattr(pred, "root_prediction", None)
    out = {"label": None, "probs": None, "regression": {}}
    if root is None:
        return out
    for node in [root] + (root.walk() if hasattr(root, "walk") else []):
        cls = getattr(node, "classification", None)
        if cls is not None and out["label"] is None:
            lab = getattr(cls, "predicted_label", None) or getattr(cls, "label", None)
            pr = getattr(cls, "class_probabilities", None) or getattr(cls, "probabilities", None)
            out["label"] = None if lab is None else str(lab)
            if isinstance(pr, dict):
                out["probs"] = {str(k): float(v) for k, v in pr.items()}
        reg = getattr(node, "regression", None)
        for k, v in (getattr(reg, "values", None) or {}).items():
            try:
                out["regression"][str(k)] = float(v)
            except (TypeError, ValueError):
                pass
    return out

# The kept fields on every result row (adds "model" so the full-batch routing is recorded per run).
KEEP = ("dataset", "task", "tier", "subject", "model", "outputs", "ground_truth", "diagnosis")

def build_jobs(dkey, tiers):
    "SUBSET jobs: the 2 built subjects x the requested tiers, all on DEFAULT_MODEL (single-provider sanity)."
    d = DATASETS[dkey]; jobs = []
    for task in d["tasks"]:
        for subj in d["subjects"]:
            for tier in tiers:
                src = d["src_dir"](tier, subj, task["name"])
                jobs.append(dict(dataset=dkey, task=task["name"], tier=tier, subject=subj, model=DEFAULT_MODEL,
                                 src_dir=src, spec=task["spec"], gi=task["gi"], outputs=task["outputs"],
                                 target_label=task["target_label"], control_label=task["control_label"],
                                 valid_ranges=task.get("valid_ranges", {}), valid_labels=task.get("valid_labels"),
                                 ground_truth=d["ground_truth"](subj, task["name"]), diagnosis=d["diagnosis"](subj),
                                 uid=f"{dkey}__{task['name']}__{tier}__{subj}".replace("/", "-")))
    return jobs

def build_design_jobs(dkey, seed=DESIGN_SEED):
    "FULL-batch jobs from the seeded design: one tier + one provider per subject (both optimizations)."
    d = DATASETS[dkey]; jobs = []
    resolve = d.get("full_src_dir", d["src_dir"])
    for a in assign_design(d["full_cohort"], d["tiers"], seed):
        subj, tier, model = a["subject"], a["tier"], a["model"]
        for task in d["tasks"]:                                   # numeracy runs both phenotypes at the assigned tier
            src = resolve(tier, subj, task["name"])
            jobs.append(dict(dataset=dkey, task=task["name"], tier=tier, subject=subj, model=model,
                             src_dir=src, spec=task["spec"], gi=task["gi"], outputs=task["outputs"],
                             target_label=task["target_label"], control_label=task["control_label"],
                             valid_ranges=task.get("valid_ranges", {}), valid_labels=task.get("valid_labels"),
                             ground_truth=d["full_gt"](subj, task["name"]), diagnosis=d["full_diag"](subj),
                             uid=f"{dkey}__{task['name']}__{tier}__{subj}__{model}".replace("/", "-")))
    return jobs

_PANEL_PREFLIGHT = {"checked_at": 0.0, "results": []}

def verify_model_panel(force=False, ttl_s=900):
    """Make one tiny real completion with every full-batch model; fail before a costly batch if any is down."""
    global _PANEL_PREFLIGHT
    age = time.time() - _PANEL_PREFLIGHT["checked_at"]
    if not force and _PANEL_PREFLIGHT["results"] and age < ttl_s:
        results = _PANEL_PREFLIGHT["results"]
        print(f"provider preflight: using {age:.0f}s-old live results ({sum(r['ok'] for r in results)}/{len(results)} ready)")
    else:
        from src.full_stack.backend.utils.llm_client import get_llm_client
        client = get_llm_client().client
        results = []
        print("provider preflight: one minimal live completion per full-batch model", flush=True)
        for m in MODEL_PANEL:
            t0 = time.time(); error = None; content = ""
            for attempt in range(1, 3):
                try:
                    response = client.chat.completions.create(
                        model=m["id"], messages=[{"role": "user", "content": "Reply with only: OK"}],
                        max_completion_tokens=256, temperature=0, timeout=60,
                        extra_body={"reasoning": {"effort": "minimal"}})
                    content = (response.choices[0].message.content or "").strip() if response.choices else ""
                    if not content:
                        raise ValueError("empty completion")
                    error = None; break
                except Exception as e:
                    error = f"{type(e).__name__}: {e}"
                    if attempt < 2:
                        time.sleep(RETRY_BACKOFF_S)
            result = dict(model=m["id"], host=m["host"], ok=error is None,
                          seconds=round(time.time() - t0, 1), reply=content[:40], error=error)
            results.append(result)
            state = f"OK ({result['reply']!r})" if result["ok"] else f"FAILED {error[:180]}"
            print(f"  {m['id']:32} {m['host']:9} {result['seconds']:>5.1f}s  {state}", flush=True)
        _PANEL_PREFLIGHT = {"checked_at": time.time(), "results": results}
    failed = [r for r in results if not r["ok"]]
    if failed:
        raise RuntimeError("Provider preflight failed: " + "; ".join(f"{r['model']}: {r['error']}" for r in failed))
    print(f"provider preflight PASSED: all {len(results)} models returned a live completion.", flush=True)
    return results

RUN_TIMEOUT_S = 1500   # watchdog: a single attempt may not exceed this (keeps the batch from ever hanging)

class _CaptureAndStream:
    """Tee engine output to the notebook in real time while retaining it for failure diagnostics."""
    def __init__(self, capture, live):
        self.capture, self.live = capture, live
    def write(self, text):
        self.capture.write(text)
        try:
            self.live.write(text); self.live.flush()
        except Exception:
            pass
        return len(text)
    def flush(self):
        self.capture.flush()
        try:
            self.live.flush()
        except Exception:
            pass
    def isatty(self):
        return False

def _validate_harvested_prediction(job, prediction):
    """Reject missing, non-finite, off-scale, or invalid-class outputs before they enter the cache."""
    regression = prediction.get("regression") or {}
    for output in job["outputs"]:
        value = regression.get(output)
        if value is None or not np.isfinite(value):
            raise ValueError(f"missing/non-finite required prediction: {output}={value}")
        bounds = (job.get("valid_ranges") or {}).get(output)
        if bounds and not (float(bounds[0]) <= float(value) <= float(bounds[1])):
            raise ValueError(f"off-scale prediction: {output}={value} outside [{bounds[0]}, {bounds[1]}]")
    labels = job.get("valid_labels")
    if labels and prediction.get("label") not in labels:
        raise ValueError(f"invalid classification label: {prediction.get('label')!r}; expected one of {labels}")
    return prediction

def _run_one(job, live_out=None):
    buf, t0, pdir = io.StringIO(), time.time(), None
    try:
        from main import run_compass_pipeline
        set_model(job.get("model") or DEFAULT_MODEL)              # route THIS run to its assigned provider
        pdir = SCRATCH / "inputs" / job["uid"]
        pdir.mkdir(parents=True, exist_ok=True)
        for f in ("data_overview.json", "hierarchical_deviation_map.json", "multimodal_data.json", "non_numerical_data.txt"):
            shutil.copy(job["src_dir"] / f, pdir / f)
        stream_logs = bool(job.get("_stream_logs", True) and live_out is not None)
        stdout_target = _CaptureAndStream(buf, live_out) if stream_logs else buf
        stderr_target = _CaptureAndStream(buf, live_out) if stream_logs else buf
        with contextlib.redirect_stdout(stdout_target), contextlib.redirect_stderr(stderr_target):
            res = run_compass_pipeline(participant_dir=pdir, target_condition=job["target_label"],
                control_condition=job["control_label"], prediction_task_spec=job["spec"],
                agent_instructions={"global": job["gi"]}, max_iterations=MAX_ITERATIONS,
                verbose=stream_logs, interactive_ui=False)
        prediction = _validate_harvested_prediction(job, harvest(res))
        return {**{k: job[k] for k in KEEP}, "prediction": prediction,
                "seconds": round(time.time() - t0, 1), "ok": True}
    except Exception as e:
        return {**{k: job[k] for k in KEEP}, "error": f"{type(e).__name__}: {e}",
                "log_tail": buf.getvalue()[-2400:], "seconds": round(time.time() - t0, 1), "ok": False}
    finally:
        if pdir is not None:
            shutil.rmtree(pdir, ignore_errors=True)

def run_job(job):
    "Run one participant attempt with a watchdog and visible liveness heartbeats."
    import threading
    progress_out, box, started = sys.stdout, {}, time.time()
    if job.get("_stream_logs", True):
        print("      ---------- LIVE COMPASS INFERENCE OUTPUT ----------", file=progress_out, flush=True)
    th = threading.Thread(target=lambda: box.__setitem__("r", _run_one(job, progress_out)), daemon=True)
    th.start()
    while th.is_alive():
        remaining = RUN_TIMEOUT_S - (time.time() - started)
        if remaining <= 0:
            break
        th.join(timeout=min(RUN_HEARTBEAT_S, remaining))
        if th.is_alive() and time.time() - started < RUN_TIMEOUT_S:
            print(f"      ... still running | elapsed {_fmt_duration(time.time()-started)} | "
                  f"attempt {job.get('_attempt', 1)}/{JOB_MAX_ATTEMPTS}", file=progress_out, flush=True)
    if th.is_alive():
        return {**{k: job[k] for k in KEEP}, "error": f"timeout after {RUN_TIMEOUT_S}s",
                "seconds": round(time.time() - started, 1), "timed_out": True, "ok": False}
    return box.get("r") or {**{k: job[k] for k in KEEP}, "error": "worker exited without a result",
                            "seconds": round(time.time() - started, 1), "ok": False}

def _job_key(r):
    return (r["subject"], r["task"], r["tier"], r.get("model") or DEFAULT_MODEL)

def _fmt_duration(seconds):
    seconds = max(0, int(seconds or 0)); minutes, seconds = divmod(seconds, 60); hours, minutes = divmod(minutes, 60)
    return f"{hours}h {minutes:02d}m" if hours else (f"{minutes}m {seconds:02d}s" if minutes else f"{seconds}s")

def _error_summary(r, limit=600):
    return " ".join(str(r.get("error", "unknown error")).split())[:limit]

def _print_failure(r, prefix="      "):
    print(f"{prefix}failure: {_error_summary(r)}", flush=True)
    tail = [line.strip() for line in str(r.get("log_tail", "")).splitlines() if line.strip()][-8:]
    if tail:
        print(f"{prefix}engine log tail:", flush=True)
        for line in tail:
            print(f"{prefix}  {line[:300]}", flush=True)

def _tag(r):
    if not r.get("ok"):
        return "ERROR " + _error_summary(r, 120)
    p = r["prediction"]
    if p["label"] is not None:
        pr = max(p["probs"].values()) if p.get("probs") else None
        return f"dx={p['label'][:20]}" + (f" p={pr:.2f}" if pr else "")
    o = r["outputs"][0]; gt = (r["ground_truth"] or {}).get(o)
    return f"{o.split('__')[-1][:16]}: pred={p['regression'].get(o)}, true={gt}"

def _run_batch(dkey, jobs, label, rerun=False, max_usd=SUBSET_MAX_USD, stream_logs=True):
    """Run jobs with live engine logs, retries, exact-model resume, and an optional dollar guard."""
    if MAX_WORKERS != 1:
        raise RuntimeError("Keep MAX_WORKERS=1: per-run model routing uses process-global settings.")
    cache = SCRATCH / f"results_{dkey}_{label}.json"
    prior = [] if rerun else (json.loads(cache.read_text()) if cache.exists() else [])
    prior_by_key = {_job_key(r): r for r in prior if r.get("ok")}
    prior_failed_by_key = {_job_key(r): r for r in prior if not r.get("ok")}
    rows = [prior_by_key[_job_key(j)] for j in jobs if _job_key(j) in prior_by_key]
    # Keep only exact current-plan successes; cached failures and stale model assignments are retried/ignored.
    done = {_job_key(r) for r in rows}
    todo = [j for j in jobs if _job_key(j) not in done]
    print(f"[{dkey}] {label}: {len(jobs)} total | {len(rows)} cached ok | {len(todo)} pending | "
          f"up to {JOB_MAX_ATTEMPTS} attempts each", flush=True)
    if not todo:
        print(f"[{dkey}] nothing to do; using cached results (set rerun=True to redo).")
        return rows
    usage_start = openrouter_usage()
    batch_started = time.time(); new_ok = attempts_run = completed = 0; stop = False
    cap_text = "NONE (usage reporting only)" if max_usd is None else f"+${max_usd:.2f}"
    print(f"[{dkey}] sequential outer runner | internal executor fan-out={os.environ['COMPASS_EXECUTOR_MAX_WORKERS']} | "
          f"spend cap {cap_text} | engine output {'LIVE/VERBOSE' if stream_logs else 'captured'}", flush=True)
    for job in todo:
        previous_failure = prior_failed_by_key.get(_job_key(job), {})
        failures = list(previous_failure.get("attempt_errors") or [])
        attempt_seconds = [float(previous_failure.get("total_seconds", 0) or 0)] if previous_failure else []
        previous_attempts = int(previous_failure.get("attempts", len(failures)) or len(failures))
        for attempt in range(1, JOB_MAX_ATTEMPTS + 1):
            attempts_run += 1
            run_job_payload = {**job, "_attempt": attempt, "_stream_logs": stream_logs}
            print(f"  START {completed+1:>3}/{len(todo)} | {job['tier']} | {job['subject']} | "
                  f"{job['task']} | {job['model']} | attempt {attempt}/{JOB_MAX_ATTEMPTS}", flush=True)
            r = run_job(run_job_payload); attempt_seconds.append(float(r.get("seconds", 0) or 0))
            if stream_logs:
                print("      ---------- END LIVE COMPASS OUTPUT ----------", flush=True)
            usage_now = openrouter_usage()
            spend_hit = (max_usd is not None and usage_start is not None and usage_now is not None
                         and usage_now - usage_start >= max_usd)
            if not r.get("ok"):
                failures.append(dict(attempt=previous_attempts + attempt, seconds=r.get("seconds"), error=r.get("error"),
                                     log_tail=r.get("log_tail", "")))
                _print_failure(r)
                if r.get("timed_out"):
                    print("      watchdog timeout: stopping the batch; restart the kernel before continuing so the "
                          "timed-out daemon cannot overlap another model route.", flush=True)
                elif not spend_hit and attempt < JOB_MAX_ATTEMPTS:
                    print(f"      automatic whole-pipeline retry in {RETRY_BACKOFF_S}s (same assigned model).", flush=True)
                    time.sleep(RETRY_BACKOFF_S)
                    continue
            r["attempts"] = previous_attempts + attempt
            r["attempt_errors"] = failures
            r["total_seconds"] = round(sum(attempt_seconds), 1)
            rows.append(r); completed += 1; new_ok += 1 if r.get("ok") else 0
            cache.write_text(json.dumps(rows, indent=2, default=str))   # persist after EACH terminal job
            elapsed = time.time() - batch_started
            eta = elapsed / completed * (len(todo) - completed) if completed else 0
            recovered = f" | recovered on cumulative attempt {r['attempts']}" if r.get("ok") and r["attempts"] > 1 else ""
            print(f"  DONE  {completed:>3}/{len(todo)} | {_fmt_duration(r['total_seconds'])} | ETA {_fmt_duration(eta)} | "
                  f"{_tag(r)}{recovered}", flush=True)
            if spend_hit:
                print(f"  [spend guard hit +${usage_now-usage_start:.2f}; progress saved. Re-run this cell to continue.]", flush=True)
                stop = True
            if r.get("timed_out"):
                stop = True
            break
        if stop:
            break
    usage_end = openrouter_usage()
    spent = (max(0.0, usage_end - usage_start)
             if usage_start is not None and usage_end is not None else None)
    ok = [r for r in rows if r.get("ok")]
    (SCRATCH / f"results_{dkey}_{label}_meta.json").write_text(json.dumps({
        "n_runs": len(rows), "n_ok": len(ok), "attempts_this_invocation": attempts_run,
        "spend_cap_usd": max_usd, "verbose_streaming": bool(stream_logs),
        "avg_seconds": round(float(np.mean([r.get("total_seconds", r["seconds"]) for r in ok])), 1) if ok else None,
        "cost_per_new_success": round(spent / new_ok, 5) if spent is not None and new_ok else None,
        "spent_usd_this_run": round(spent, 5) if spent is not None else None}, indent=2))
    spent_text = f"${spent:.4f}" if spent is not None else "unavailable"
    print(f"[{dkey}] {label}: {len(ok)}/{len(rows)} terminal rows ok | {attempts_run} attempt(s) this invocation | "
          f"spent {spent_text} | elapsed {_fmt_duration(time.time()-batch_started)}", flush=True)
    return rows

def run_dataset(dkey, tiers, rerun=False, label="subset", verbose=True):
    "SUBSET runner (unchanged interface): the 2 built subjects on `tiers`, all on DEFAULT_MODEL."
    return _run_batch(dkey, build_jobs(dkey, tiers), label, rerun, max_usd=SUBSET_MAX_USD, stream_logs=verbose)

def run_full_design(dkey, rerun=False, label="full", verbose=True):
    "FULL-batch runner: live verbose output, no dollar cap, seeded 5-model design, missing-input skip."
    if dkey == "PSYCHOSIS":
        from validation.common.openneuro_full_inputs import ensure_psychosis_full_inputs
        ensure_psychosis_full_inputs(REPO, SCRATCH, progress=lambda s: print(s, flush=True))
    verify_model_panel()                                         # tiny live fail-fast check, cached for 15 minutes
    jobs = build_design_jobs(dkey)
    runnable = [j for j in jobs if j["src_dir"].is_dir()]
    missing = len(jobs) - len(runnable)
    if missing:
        print(f"[{dkey}] {missing}/{len(jobs)} design runs have no blinded inputs on disk yet "
              f"(build them via the dataset pipeline); running the {len(runnable)} that exist.")
    return _run_batch(dkey, runnable, label, rerun, max_usd=None, stream_logs=verbose)

# Load-check every subset record through the engine DataLoader (cheap, no spend).
from src.full_stack.backend.utils.core.data_loader import DataLoader
dl = DataLoader(); checked = 0
for dkey, d in DATASETS.items():
    for job in build_jobs(dkey, [d["subset_tier"]]):
        assert job["src_dir"].is_dir(), f"missing input dir: {job['src_dir']}"
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            data = dl.load(job["src_dir"])
        cov = data.data_overview
        assert cov.domain_coverage and cov.total_tokens and cov.total_tokens > 0, f"bad overview: {job['src_dir']}"
        blob = json.dumps(getattr(data.multimodal_data, "raw", None) or getattr(data.multimodal_data, "__dict__", {}), default=str)
        for tok in job["outputs"]:
            assert tok not in blob, f"target {tok} leaked into inputs: {job['src_dir']}"
        checked += 1
def _reg_frame(rows):
    "Tidy (subject, tier, output, predicted, truth, error) frame from a dataset's results."
    out = []
    for r in rows:
        if not r.get("ok"):
            continue
        for o in r["outputs"]:
            pred = r["prediction"]["regression"].get(o); gt = (r["ground_truth"] or {}).get(o)
            out.append(dict(subject=r["subject"], tier=r["tier"], output=o, predicted=pred, truth=gt,
                            error=None if (pred is None or gt is None) else pred - gt))
    return pd.DataFrame(out)

RUN = {}   # dataset key -> subset results (filled by the per-dataset cells below)
print(f"OK: {checked} full-tier subset records load cleanly through the engine, 0 target leaks.")

In [ ]:
# ---- Subset smoke test: 2 subjects per dataset, each on ONE random data tier (seeded), on the deepseek
# anchor. A quick pre-flight to eyeball that predictions are sensible before committing to the full batch. ----
def run_smoketest(dkey, rerun=False, verbose=True):
    "Run 2 subset subjects with full live COMPASS inference output by default."
    d = DATASETS[dkey]
    rng = random.Random(f"{DESIGN_SEED}:{dkey}")   # deterministic per-dataset seed (string seed is reproducible)
    jobs = []
    for subj in d["subjects"]:
        tier = rng.choice(d["tiers"])                            # one random data tier for this subject
        for task in d["tasks"]:
            src = d["src_dir"](tier, subj, task["name"])
            jobs.append(dict(dataset=dkey, task=task["name"], tier=tier, subject=subj, model=DEFAULT_MODEL,
                             src_dir=src, spec=task["spec"], gi=task["gi"], outputs=task["outputs"],
                             target_label=task["target_label"], control_label=task["control_label"],
                             ground_truth=d["ground_truth"](subj, task["name"]), diagnosis=d["diagnosis"](subj),
                             uid=f"{dkey}__{task['name']}__{tier}__{subj}".replace("/", "-")))
    return _run_batch(dkey, jobs, "smoketest", rerun, max_usd=SUBSET_MAX_USD, stream_logs=verbose)

def smoketest_table(dkey):
    "Print a clear predicted-vs-truth table for the smoke test (the assigned random tier is shown per row)."
    rows = RUN.get(dkey, [])
    n_ok = sum(1 for r in rows if r.get("ok"))
    print(f"=== {dkey}: {n_ok}/{len(rows)} runs ok  |  2 subjects, one random tier each, model {DEFAULT_MODEL} ===")
    for r in rows:
        subj = str(r["subject"])[:22]
        if not r.get("ok"):
            print(f"  {subj:22} {r['tier']:28} ERROR after {r.get('attempts', 1)} attempt(s)")
            _print_failure(r, prefix="    ")
            continue
        p = r["prediction"]; gt = r.get("ground_truth") or {}
        if p.get("label") is not None:
            ok = "correct" if p["label"] == r.get("diagnosis") else "WRONG"
            print(f"  {subj:22} {r['tier']:28} diagnosis  pred={p['label']!s:24} true={r.get('diagnosis')!s:24} [{ok}]")
        for o in r["outputs"]:
            pr = p["regression"].get(o); tr = gt.get(o)
            if pr is None:
                continue
            err = "" if tr is None else f"  err={round(pr - tr, 2)}"
            print(f"  {subj:22} {r['tier']:28} {o.split('__')[-1][:26]:26} pred={pr!s:10} true={tr!s:10}{err}")

def run_all_smoketests(rerun=False, check_panel=True, plots=True, verbose=True):
    """One-click preflight: verify all five models, then run and summarize every dataset subset."""
    if check_panel:
        verify_model_panel()
    for dkey in ("INTELLIGENCE", "PSYCHOSIS", "NUMERACY"):
        print(f"\n{'='*88}\nALL-DATASET PREFLIGHT: {dkey}\n{'='*88}", flush=True)
        RUN[dkey] = run_smoketest(dkey, rerun=rerun, verbose=verbose)
        smoketest_table(dkey)
    if plots:
        viz_intelligence(); viz_psychosis(); viz_numeracy()
    total = sum(len(RUN.get(k, [])) for k in DATASETS)
    ok = sum(sum(1 for r in RUN.get(k, []) if r.get("ok")) for k in DATASETS)
    print(f"\nALL-DATASET PREFLIGHT COMPLETE: {ok}/{total} terminal runs ok.", flush=True)
    return {k: RUN.get(k, []) for k in DATASETS}

def viz_intelligence():
    df = _reg_frame(RUN.get("INTELLIGENCE", []))
    if df.empty:
        print("Run the INTELLIGENCE smoke-test cell above first."); return
    outs = ["IST_intelligence_total", "IST_fluid", "IST_memory", "IST_crystallised"]
    fig, ax = plt.subplots(1, 1, figsize=(9, 5), constrained_layout=True)
    x = np.arange(len(outs)); w = 0.36
    for i, subj in enumerate(sorted(df["subject"].unique())):
        s = df[df["subject"] == subj].set_index("output")
        pr = [s.loc[o, "predicted"] if o in s.index else np.nan for o in outs]
        tr = [s.loc[o, "truth"] if o in s.index else np.nan for o in outs]
        ax.bar(x + (i - 0.5) * w, pr, w * 0.9, label=f"{subj} predicted", alpha=0.85)
        ax.scatter(x + (i - 0.5) * w, tr, color="k", zorder=3, marker="_", s=260,
                   label="ground truth" if i == 0 else None)
    ax.set_xticks(x); ax.set_xticklabels([o.replace("IST_", "") for o in outs])
    ax.set_ylabel("native IST points"); ax.set_title("INTELLIGENCE: predicted bars vs true (black ticks)")
    ax.legend(fontsize=8, frameon=False); plt.show()

def viz_psychosis():
    rows = RUN.get("PSYCHOSIS", [])
    if not any(r.get("ok") for r in rows):
        print("Run the PSYCHOSIS smoke-test cell above first."); return
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.4), constrained_layout=True)
    labels, correct = [], []
    for r in rows:
        if not r.get("ok"):
            continue
        pred = r["prediction"]["label"]; true = r["diagnosis"]
        labels.append(f"{str(r['subject'])[:18]}\ntrue: {true}"); correct.append(pred == true)
        ax[0].annotate(f"predicted:\n{pred}", (len(labels) - 1, 0.5), ha="center", va="center", fontsize=9)
    ax[0].bar(range(len(labels)), [1] * len(labels), color=["#59a14f" if c else "#e15759" for c in correct])
    ax[0].set_xticks(range(len(labels))); ax[0].set_xticklabels(labels, fontsize=8)
    ax[0].set_yticks([]); ax[0].set_title("Diagnosis (green=correct, red=wrong)")
    df = _reg_frame(rows)
    sym = df[df["output"] != "target__psychosis__case_control_label"].copy()
    for subj in sorted(sym["subject"].unique()):
        s = sym[sym["subject"] == subj]
        ax[1].scatter(s["truth"], s["predicted"], label=str(subj)[:18], s=40)
    vals = sym[["truth", "predicted"]].to_numpy()
    lim = [0, max(1, (np.nanmax(vals) if vals.size else 1) * 1.1)]
    ax[1].plot(lim, lim, "k--", lw=0.8, alpha=0.6); ax[1].set_xlim(lim); ax[1].set_ylim(lim)
    ax[1].set_xlabel("true score"); ax[1].set_ylabel("predicted score")
    ax[1].set_title("BPRS total + SAPS/SANS globals: pred vs true"); ax[1].legend(fontsize=8, frameon=False)
    plt.show()

def viz_numeracy():
    df = _reg_frame(RUN.get("NUMERACY", []))
    if df.empty:
        print("Run the NUMERACY smoke-test cell above first."); return
    subjects = sorted(df["subject"].unique())
    fig, ax = plt.subplots(1, 1, figsize=(9, 5), constrained_layout=True)
    x = np.arange(len(subjects)); w = 0.2
    cmap = {"approximate_numeracy": "#4e79a7", "precise_numeracy": "#e15759"}
    for j, (o, c) in enumerate(cmap.items()):
        pr = [df[(df.subject == s) & (df.output == o)]["predicted"].mean() for s in subjects]
        tr = [df[(df.subject == s) & (df.output == o)]["truth"].mean() for s in subjects]
        ax.bar(x + (j - 0.5) * w * 2, pr, w * 1.8, color=c, alpha=0.85, label=f"{o.split('_')[0]} pred")
        ax.scatter(x + (j - 0.5) * w * 2, tr, color="k", marker="_", s=200, zorder=3,
                   label="ground truth" if j == 0 else None)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xticks(x); ax.set_xticklabels(subjects); ax.set_ylabel("population Z-score")
    ax.set_title("NUMERACY: predicted bars vs true (black ticks)")
    ax.legend(fontsize=8, frameon=False); plt.show()

print("smoke-test helpers ready: run_smoketest(dkey), run_all_smoketests(), smoketest_table(dkey), viz_intelligence/psychosis/numeracy().")

## 3. INTELLIGENCE subset run (AOMIC ID1000, OpenNeuro ds003097)

Press the SUBSET cell below to run the 2-subject smoke test: each of the 2 diverse subjects (lowest and
highest IST) is run on ONE random data tier (seeded), on the deepseek anchor. The cell prints a
predicted-vs-truth table and a plot inline, so you can eyeball the results. It caches, so a re-run does not
re-spend (pass `rerun=True` to force).

**INPUT - data-complexity ladder** (5 tiers; target is never an input; subset anchor in bold)

| tier | evidence | ~tokens |
|---|---|---|
| T1_demographics | age, sex, handedness, BMI, education, SES | 45 |
| **T2_psychological (subset)** | + NEO Big Five, BIS/BAS, STAI, identity, religiosity (all self-report) | 180 |
| T3_morphometry | + FreeSurfer brain morphometry | 3063 |
| T4_multimodal_full (everything) | + movie-fMRI functional connectome | 3473 |
| T5_brain_only | brain only: morphometry + connectome (no self-report) | 3293 |

**OUTPUT - target phenotype** (hierarchical regression, native IST 2000-R points)
- `total_intelligence` (univariate): `IST_intelligence_total`
  - `ist_subscales` (multivariate): `IST_fluid`, `IST_memory`, `IST_crystallised`

**Linguistic representation:** the global instruction states the cohort context (AOMIC ID1000, 928 healthy
Dutch young adults, ds003097, normal intelligence range) and the IST native-scale meaning, so the engine
knows the population and the scale it predicts on. Predictors are grouped under a semantic ontology
(self-report constructs plus a BRAIN domain for the imaging tiers).

In [ ]:
# Version-aware setup refresh (free; never starts an engine run). This also fixes a stale live kernel.
import sys as _sys, importlib as _importlib
from pathlib import Path as _Path
_sys.path.insert(0, str(next(b for b in [_Path.cwd(), *_Path.cwd().parents] if (b / "src" / "full_stack").is_dir())))
import validation.common.nb_bootstrap as _nb_bootstrap
_nb_bootstrap = _importlib.reload(_nb_bootstrap)
_nb_bootstrap.ensure_notebook_setup(globals())

RUN["INTELLIGENCE"] = run_smoketest("INTELLIGENCE", verbose=True)   # full live inference log per subject
smoketest_table("INTELLIGENCE")
viz_intelligence()

## 4. PSYCHOSIS subset run (first-episode psychosis, OpenNeuro ds003944 + ds003947)

Press the SUBSET cell below to run the 2-subject smoke test: 1 case + 1 control, each on ONE random data
tier (seeded), on the deepseek anchor. The cell prints a predicted-vs-truth table (diagnosis plus symptom
scores) and a plot inline. It caches (pass `rerun=True` to force). The predictor inputs were rebuilt on
2026-07-23 for the restructured ontology and 4-tier ladder below.

**INPUT - data-complexity ladder** (4 tiers; subset in bold)

| tier | evidence | ~tokens |
|---|---|---|
| T1_demographic_socioeconomic | demographics + Hollingshead SES | 70 |
| **T2_clinical_profile (subset)** | + MATRICS cognition, WASI IQ, GAF/SFS observed functioning | 1185 |
| T3_multimodal_full (full) | + all 836 resting-EEG features (multimodal ceiling) | 21074 |
| T4_eeg_brain_only | 79-feature psychosis-signature resting EEG only (brain-only) | 1468 |

The brain-only tier is a curated psychosis signature (spectral slowing + alpha deficit, alpha-peak slowing,
aperiodic 1/f, signal complexity, microstate C/D, connectivity): richer than the old 29-feature lean, far
below the full 836 (which was redundant with T3).

**INPUT - predictor ontology** (3 primary domains)
- Resting EEG (836 leaves): 8 EEG feature families
- Demographics and Socio-economic Status (8 leaves)
- Global Functioning (91 leaves): MATRICS cognition, WASI intelligence, GAF ratings and the Social
  Functioning Scale, all as secondary siblings under one umbrella

**OUTPUT - target phenotype** (mixed-type hierarchy)
- `diagnosis` (binary): Control vs First-Episode Psychosis
  - `global_severity` (univariate): BPRS 19-item total (19-133)
    - `positive_symptoms` (multivariate): 4 SAPS global ratings (0-5 each)
    - `negative_symptoms` (multivariate): 5 SANS global ratings (0-5 each)

**Linguistic representation:** the global instruction states the cohort (ds003944 + ds003947, first-episode
psychosis plus matched controls, resting EEG) and each native clinical scale with its reference mean/sd.

In [ ]:
# Version-aware setup refresh (free; never starts an engine run). This also fixes a stale live kernel.
import sys as _sys, importlib as _importlib
from pathlib import Path as _Path
_sys.path.insert(0, str(next(b for b in [_Path.cwd(), *_Path.cwd().parents] if (b / "src" / "full_stack").is_dir())))
import validation.common.nb_bootstrap as _nb_bootstrap
_nb_bootstrap = _importlib.reload(_nb_bootstrap)
_nb_bootstrap.ensure_notebook_setup(globals())

RUN["PSYCHOSIS"] = run_smoketest("PSYCHOSIS", verbose=True)   # full live inference log per subject
smoketest_table("PSYCHOSIS")
viz_psychosis()

## 5. NUMERACY subset run (numeracy after stroke, OpenNeuro ds006533)

Press the SUBSET cell below to run the 2-subject smoke test: each subject on ONE random data tier (seeded),
running BOTH numeracy phenotypes so their dissociation reads on one population-Z scale, on the deepseek
anchor. The cell prints a predicted-vs-truth table and a plot inline. It caches (pass `rerun=True` to
force).

**INPUT - data-complexity ladder** (4 tiers; subset anchor in bold)

| tier | evidence | ~tokens |
|---|---|---|
| T1_demographics | age, education, imaging modality | 22 |
| T2_aphasia | + WAB-R aphasia quotient, whole-brain lesion load | 76 |
| **T3_lesion_fine (subset, everything)** | + per-parcel lesion overlap (194 features) | 3017 |
| T4_lesion_brain_only | brain only: per-parcel lesion overlap (189 features, no demographics/clinical) | 2941 |

**OUTPUT - target phenotype** (two dissociable univariate phenotypes, predicted separately)
- `approximate_numeracy`: non-symbolic Approximate Number System, population Z-score
- `precise_numeracy`: precise symbolic numeracy, population Z-score

**Linguistic representation:** the global instruction states the cohort (ds006533, 105 left-hemisphere
chronic stroke survivors) and that both scores are population Z-scores centred on the stroke cohort
(0 = cohort mean).

In [ ]:
# Version-aware setup refresh (free; never starts an engine run). This also fixes a stale live kernel.
import sys as _sys, importlib as _importlib
from pathlib import Path as _Path
_sys.path.insert(0, str(next(b for b in [_Path.cwd(), *_Path.cwd().parents] if (b / "src" / "full_stack").is_dir())))
import validation.common.nb_bootstrap as _nb_bootstrap
_nb_bootstrap = _importlib.reload(_nb_bootstrap)
_nb_bootstrap.ensure_notebook_setup(globals())

RUN["NUMERACY"] = run_smoketest("NUMERACY", verbose=True)   # full live inference log per subject/task
smoketest_table("NUMERACY")
viz_numeracy()

## 5.1 One-click preflight: all three subsets + all five providers

Run the cell below when you want one definitive preflight before the full batches. It first makes one tiny
live completion with every model in the five-provider panel, then runs the INTELLIGENCE, PSYCHOSIS, and
NUMERACY subsets in sequence (8 terminal jobs total: 2 + 2 + 4). Successful cached jobs are skipped. Every
multi-minute attempt streams the complete COMPASS inference log plus a heartbeat every 30 seconds; a hard
failure automatically retries the complete participant pipeline once on the same assigned model, then prints
the full error summary and retained engine-log tail.

The provider probe is cached for 15 minutes in the kernel and uses only a minimal `OK` completion per model.
Pass `rerun=True` to `run_all_smoketests` only when you intentionally want to re-spend on cached successes.

In [ ]:
# Version-aware setup refresh (free; never starts an engine run). This also fixes a stale live kernel.
import sys as _sys, importlib as _importlib
from pathlib import Path as _Path
_sys.path.insert(0, str(next(b for b in [_Path.cwd(), *_Path.cwd().parents] if (b / "src" / "full_stack").is_dir())))
import validation.common.nb_bootstrap as _nb_bootstrap
_nb_bootstrap = _importlib.reload(_nb_bootstrap)
_nb_bootstrap.ensure_notebook_setup(globals())

ALL_SMOKETEST_RESULTS = run_all_smoketests(verbose=True)   # provider check + all 3 subsets, full live logs

## 6. Full-batch API cost and wall-time estimate

This estimates what the full validation (every subject in the section 0.5 design) would cost and take. It
starts from the real per-run cost and duration measured on each dataset's subset tier during the pre-run
(baked in below as `BAKED_ANCHORS`, so the estimate reproduces even after the system-temp cache is cleared;
a fresh subset meta overrides them). From that anchor:

- **Cost** scales the measured `$/run` by each tier's input-token ratio, sub-linearly (floored and capped),
  because a run's cost is mostly fixed overhead (system prompts, orchestration, reasoning), and multiplies
  each run by its assigned provider's blended price. It is computed from the exact seeded assignment section
  7 would run: one balanced tier per subject, five providers balanced within each tier.
- **Wall time** uses the flat measured seconds/run per dataset at 1 outer worker. Keep the notebook's outer
  worker count at 1 because per-run model routing is process-global; the engine still parallelizes tool calls
  internally. The provider panel spreads those sequential runs across hosts and avoids one-host dependence.

All complete inputs used by the design are available: 100 leakage-safe intelligence evaluation records,
143 psychosis recordings generated into system temp, and 105 numeracy subjects for both tasks.

In [ ]:
def tier_tokens(dkey, tier):
    "Input token budget of one subject's record at a tier (0 if unavailable), read from a built subset record."
    d = DATASETS[dkey]
    for task in d["tasks"]:
        for subj in d["subjects"]:
            p = d["src_dir"](tier, subj, task["name"]) / "data_overview.json"
            if p.exists():
                return json.loads(p.read_text()).get("total_tokens", 0)
    return 0

# Per-run cost + duration measured on each dataset's SUBSET tier during the pre-run (deepseek-v4-flash).
# Baked so this estimate reproduces even if the system-temp cache was cleared; a fresh subset meta wins.
BAKED_ANCHORS = {"INTELLIGENCE": (0.0128, 811.0), "PSYCHOSIS": (0.0349, 730.0), "NUMERACY": (0.0283, 273.0)}
FLOOR, CAP = 0.6, 3.0

def _anchor(dkey):
    meta_path = SCRATCH / f"results_{dkey}_subset_meta.json"
    if meta_path.exists():
        m = json.loads(meta_path.read_text())
        if m.get("cost_per_run") and m.get("avg_seconds"):
            return float(m["cost_per_run"]), float(m["avg_seconds"])
    return BAKED_ANCHORS[dkey]

rows = []
for dkey, d in DATASETS.items():
    cpr, avg_s = _anchor(dkey)
    anchor_tok = max(1, tier_tokens(dkey, d["subset_tier"]))
    def cost_scale(t):
        return max(FLOOR, min(CAP, tier_tokens(dkey, t) / anchor_tok))
    n_tasks = len(d["tasks"])
    design = assign_design(d["full_cohort"], d["tiers"])          # one balanced tier + provider per subject
    runs = len(design) * n_tasks
    cost = sum(n_tasks * cpr * cost_scale(a["tier"]) * model_price_factor(a["model"]) for a in design)
    rows.append(dict(dataset=dkey, cohort=d["n_cohort"], tiers=len(d["tiers"]), tasks=n_tasks,
                     runs=runs, est_cost_usd=round(cost, 2),
                     est_wall_h=round(runs * avg_s / MAX_WORKERS / 3600, 1)))

est = pd.DataFrame(rows)
print("Full-validation estimate (between-subjects design, five-provider panel):\n")
print(est.to_string(index=False))
print(f"\nTOTAL: ${est['est_cost_usd'].sum():.2f} over {int(est['runs'].sum())} runs, "
      f"~{est['est_wall_h'].sum():.0f} h at {MAX_WORKERS} worker.")
print("Each subject is run at one balanced data tier, with the five providers balanced within each tier. "
      "Wall time is at 1 outer worker; keep MAX_WORKERS=1 because model routing is process-global.")

In [ ]:
# The exact assignment the full batch would run: each subject sits in one (tier, provider) cell. The tiers
# are balanced across subjects (optimization 2) and, within every tier, the five providers are balanced
# (optimization 1), so tier and provider are decorrelated. Reproducible from DESIGN_SEED.
for dkey, d in DATASETS.items():
    design = assign_design(d["full_cohort"], d["tiers"])
    tab = design_balance_table(design, d["tiers"])
    print(f"=== {dkey}: between-subjects tier x provider plan (n={len(design)} subjects, "
          f"{len(d['tasks'])} task(s) each) ===")
    print(tab.to_string()); print()

### 6.1 The nested balanced design (visual)

The heatmaps below draw the seeded plan the full batch will run: for each dataset, rows are the
data-complexity tiers and columns are the five providers, and each cell is how many subjects fall in that
tier and provider pair. Even shading with equal row and column totals is the balanced crossing (providers
balanced within every tier, tiers balanced across subjects), so provider is not confounded with tier. This
is drawn before any run.

In [ ]:
from validation.common import batch_report as BR
REPORTS = SCRATCH / "reports"

def load_full(dkey):
    "Cached full-batch results for a dataset (from disk if present, else this session's RUN dict)."
    f = SCRATCH / f"results_{dkey}_full.json"
    if f.exists():
        return json.loads(f.read_text())
    return RUN.get(f"{dkey}_full", [])

# Draw the seeded plan the full batch will run: tier x provider subject counts, one panel per dataset.
_designs = {dk: assign_design(DATASETS[dk]["full_cohort"], DATASETS[dk]["tiers"]) for dk in DATASETS}
_ = BR.plot_design(_designs, [m["id"] for m in MODEL_PANEL])

## 7. Full-batch runs (the section 0.5 design; guarded manual entrypoints)

Each cell runs one dataset's full cohort under the section 0.5 design (one balanced tier and provider per subject): `run_full_design` builds the seeded
plan (one tier per subject via optimization 2, one balanced provider per subject via optimization 1) and
executes it, routing every run to its assigned model. They are guarded by `RUN_FULL_BATCH` so they never
fire by accident; read the section 6 estimate, set the flag to `True`, and run one dataset at a time.
Before the first cohort job, all five models must pass a tiny live-completion preflight (cached for 15
minutes). Each hard participant failure retries once automatically on its assigned model, with 30-second
heartbeats, ETA, spend reporting, and detailed terminal errors printed inline. Full batches intentionally
have **no dollar spend cap**; `RUN_FULL_BATCH` remains the explicit safety switch.
Results cache to system temp under `results_<dataset>_full.json` and are resumable. The full design uses
all complete inputs: 100 intelligence jobs, 143 psychosis jobs, and 210 numeracy jobs (105 subjects x 2
tasks), for 453 jobs total.\n\nAfter each dataset's run cell is an ANALYSIS cell (`BR.report(...)`) that quantifies per-output, per-tier, and cross-provider performance (regression MAE/RMSE/bias/correlations; diagnosis accuracy/balanced accuracy/AUROC) and draws a performance multiplot, saving the tables and figure under the scratch `reports/` folder. Those analysis cells are safe to run any time: before a batch they just report that no results exist yet. The design visual for all three datasets is in section 6.1.

In [ ]:
RUN_FULL_BATCH = False   # set True to actually spend on a full-cohort run (the section 0.5 design)

_dk = "INTELLIGENCE"
if RUN_FULL_BATCH:
    RUN[f"{_dk}_full"] = run_full_design(_dk, label="full", verbose=True)   # NO dollar cap; live engine log
else:
    _d = DATASETS[_dk]
    print(f"[{_dk}] full batch GUARDED (RUN_FULL_BATCH=False). With the flag True this would run:")
    print(f"  {_d['n_cohort']} subjects x {len(_d['tasks'])} task = {_d['n_cohort']*len(_d['tasks'])} runs, "
          f"one balanced tier + provider per subject across {len(_d['tiers'])} tiers (between-subjects).")
    print("  OUTPUT: total intelligence (univariate) then 3 IST subscales (multivariate). Cost: section 6.")

In [ ]:
BR.report("INTELLIGENCE", load_full("INTELLIGENCE"), out_dir=REPORTS)   # per-tier + cross-provider performance (run the full-batch cell above first)

In [ ]:
_dk = "PSYCHOSIS"
if RUN_FULL_BATCH:
    RUN[f"{_dk}_full"] = run_full_design(_dk, label="full", verbose=True)   # NO dollar cap; live engine log
else:
    _d = DATASETS[_dk]
    print(f"[{_dk}] full batch GUARDED (RUN_FULL_BATCH=False). With the flag True this would run:")
    print(f"  {_d['n_cohort']} subjects x {len(_d['tasks'])} task = {_d['n_cohort']*len(_d['tasks'])} runs, "
          f"one balanced tier + provider per subject across {len(_d['tiers'])} tiers (between-subjects).")
    print("  OUTPUT: diagnosis (binary) -> BPRS total -> SAPS/SANS globals. Cost: section 6.")

In [ ]:
BR.report("PSYCHOSIS", load_full("PSYCHOSIS"), out_dir=REPORTS)   # per-tier + cross-provider performance (run the full-batch cell above first)

In [ ]:
_dk = "NUMERACY"
if RUN_FULL_BATCH:
    RUN[f"{_dk}_full"] = run_full_design(_dk, label="full", verbose=True)   # NO dollar cap; live engine log
else:
    _d = DATASETS[_dk]
    print(f"[{_dk}] full batch GUARDED (RUN_FULL_BATCH=False). With the flag True this would run:")
    print(f"  {_d['n_cohort']} subjects x {len(_d['tasks'])} tasks = {_d['n_cohort']*len(_d['tasks'])} runs, "
          f"one balanced tier + provider per subject across {len(_d['tiers'])} tiers (between-subjects).")
    print("  OUTPUT: approximate + precise numeracy (two dissociable univariate phenotypes). Cost: section 6.")

In [ ]:
BR.report("NUMERACY", load_full("NUMERACY"), out_dir=REPORTS)   # per-tier + cross-provider performance (run the full-batch cell above first)

## 8. Summary

In [ ]:
allrows = [r for k in ("INTELLIGENCE", "PSYCHOSIS", "NUMERACY") for r in RUN.get(k, [])]
ok = [r for r in allrows if r.get("ok")]
print(f"subset engine runs: {len(allrows)} total | {len(ok)} ok | {len(allrows)-len(ok)} errored")
if ok:
    print(f"avg run time: {np.mean([r['seconds'] for r in ok]):.0f}s | subset anchor model: {DEFAULT_MODEL}")
    frames = [_reg_frame(RUN.get(k, [])) for k in ("INTELLIGENCE", "PSYCHOSIS", "NUMERACY")]
    allreg = pd.concat([f for f in frames if not f.empty], ignore_index=True) if any(not f.empty for f in frames) else pd.DataFrame()
    if not allreg.empty:
        allreg["abserr"] = allreg["error"].abs()
        print("\nmean absolute error by output:")
        print(allreg.dropna(subset=["abserr"]).groupby("output")["abserr"].mean().round(2).to_string())
for r in [r for r in allrows if not r.get("ok")][:8]:
    print(f"  ERROR {r['dataset']} {r['tier']} {r['subject']}: {r.get('error')}")
print("\nPress each dataset's SUBSET cell above to run its 2-subject smoke test (each subject on one random "
      "tier). The full validation (section 0.5) contains 453 jobs: 100 intelligence, 143 psychosis, and "
      "210 numeracy (105 subjects x 2 tasks), balanced by tier and provider; see the estimate in section 6 "
      "and the guarded manual entrypoints in section 7.")